# MascotaLM — Tu Mascota Virtual

Entrena un LLM de ~9M parámetros que habla como Mochi, una pequeña mascota virtual tipo Tamagotchi.

**Qué hace este notebook:**
1. Descarga el dataset de 60K conversaciones desde HuggingFace (`Brax055/Mascota_virtual`)
2. Entrena un tokenizer BPE sobre los datos
3. Entrena un transformer de 6 capas (8.7M parámetros)
4. Prueba el modelo con conversaciones de ejemplo

**Arquitectura:** 6 capas, 384 dim, 6 cabezas, ReLU FFN, LayerNorm, vocab 4096

**Ejecutar en Google Colab con GPU T4 (gratis) — tarda ~2 minutos.**

## 1. Configuración

Instalar dependencias y crear directorio de trabajo.

In [ ]:
!pip install -q torch tokenizers tqdm numpy datasets huggingface_hub

import torch
print(f'PyTorch {torch.__version__}')
print(f'CUDA: {torch.cuda.is_available()}')
if torch.cuda.is_available():
    print(f'GPU: {torch.cuda.get_device_name(0)}')

In [ ]:
import os, shutil

# Empezar limpio — borra archivos de runs anteriores
if os.path.exists('/content/mascota'):
    shutil.rmtree('/content/mascota')
os.makedirs('/content/mascota')
os.chdir('/content/mascota')
print(f'Directorio de trabajo: {os.getcwd()}')

## 2. Archivos Fuente

Escribir el código del modelo al disco. Estos son los únicos archivos necesarios:
- `config.py` — hiperparámetros del modelo y entrenamiento
- `model.py` — arquitectura transformer
- `dataset.py` — carga de datos y batching
- `train.py` — loop de entrenamiento
- `inference.py` — interfaz de chat

In [ ]:
%%writefile config.py
"""MascotaLM — configuración del modelo."""

from dataclasses import dataclass


@dataclass
class MascotaConfig:
    vocab_size: int = 4096
    max_seq_len: int = 128
    d_model: int = 384
    n_layers: int = 6
    n_heads: int = 6
    ffn_hidden: int = 768
    dropout: float = 0.1

    # Tokens especiales
    pad_id: int = 0
    bos_id: int = 1           # <|im_start|>
    eos_id: int = 2           # <|im_end|>


# Alias para compatibilidad con código que usa GuppyConfig
GuppyConfig = MascotaConfig


@dataclass
class TrainConfig:
    batch_size: int = 32
    learning_rate: float = 3e-4
    min_lr: float = 3e-5
    weight_decay: float = 0.1
    warmup_steps: int = 200
    max_steps: int = 10000
    eval_interval: int = 200
    save_interval: int = 500
    grad_clip: float = 1.0
    device: str = "auto"
    seed: int = 42
    data_dir: str = "data"
    output_dir: str = "checkpoints"


In [ ]:
%%writefile model.py
"""
MascotaLM — cerebrito de mascota virtual.

Transformer vanilla: multi-head attention, ReLU FFN, LayerNorm, positional embeddings aprendidos.
Sin GQA, sin SwiGLU, sin parallel residual, sin RoPE. Lo más simple posible.
"""

import math
import torch
import torch.nn as nn
import torch.nn.functional as F
from config import MascotaConfig


class Attention(nn.Module):
    def __init__(self, config):
        super().__init__()
        self.n_heads = config.n_heads
        self.head_dim = config.d_model // config.n_heads

        self.qkv = nn.Linear(config.d_model, 3 * config.d_model)
        self.out = nn.Linear(config.d_model, config.d_model)
        self.dropout = nn.Dropout(config.dropout)

    def forward(self, x, mask=None):
        B, T, C = x.shape
        qkv = self.qkv(x).reshape(B, T, 3, self.n_heads, self.head_dim).permute(2, 0, 3, 1, 4)
        q, k, v = qkv[0], qkv[1], qkv[2]

        attn = (q @ k.transpose(-2, -1)) / math.sqrt(self.head_dim)
        if mask is not None:
            attn = attn.masked_fill(mask == 0, float("-inf"))
        attn = self.dropout(F.softmax(attn, dim=-1))
        return self.out((attn @ v).transpose(1, 2).contiguous().view(B, T, C))


class FFN(nn.Module):
    def __init__(self, config):
        super().__init__()
        self.up = nn.Linear(config.d_model, config.ffn_hidden)
        self.down = nn.Linear(config.ffn_hidden, config.d_model)
        self.dropout = nn.Dropout(config.dropout)

    def forward(self, x):
        return self.dropout(self.down(F.relu(self.up(x))))


class Block(nn.Module):
    def __init__(self, config):
        super().__init__()
        self.norm1 = nn.LayerNorm(config.d_model)
        self.attn = Attention(config)
        self.norm2 = nn.LayerNorm(config.d_model)
        self.ffn = FFN(config)

    def forward(self, x, mask=None):
        x = x + self.attn(self.norm1(x), mask)
        x = x + self.ffn(self.norm2(x))
        return x


class MascotaLM(nn.Module):
    def __init__(self, config: MascotaConfig):
        super().__init__()
        self.config = config

        self.tok_emb = nn.Embedding(config.vocab_size, config.d_model)
        self.pos_emb = nn.Embedding(config.max_seq_len, config.d_model)
        self.drop = nn.Dropout(config.dropout)
        self.blocks = nn.ModuleList([Block(config) for _ in range(config.n_layers)])
        self.norm = nn.LayerNorm(config.d_model)
        self.lm_head = nn.Linear(config.d_model, config.vocab_size, bias=False)
        self.lm_head.weight = self.tok_emb.weight  # tie weights

        self.apply(self._init_weights)

    def _init_weights(self, m):
        if isinstance(m, nn.Linear):
            nn.init.normal_(m.weight, mean=0.0, std=0.02)
            if m.bias is not None:
                nn.init.zeros_(m.bias)
        elif isinstance(m, nn.Embedding):
            nn.init.normal_(m.weight, mean=0.0, std=0.02)

    def forward(self, idx, targets=None):
        B, T = idx.shape
        pos = torch.arange(T, device=idx.device)
        x = self.drop(self.tok_emb(idx) + self.pos_emb(pos))
        mask = torch.tril(torch.ones(T, T, device=idx.device)).unsqueeze(0).unsqueeze(0)

        for block in self.blocks:
            x = block(x, mask)

        logits = self.lm_head(self.norm(x))

        loss = None
        if targets is not None:
            loss = F.cross_entropy(
                logits.view(-1, self.config.vocab_size),
                targets.view(-1),
                ignore_index=0,
            )

        return logits, loss

    @torch.no_grad()
    def generate(self, idx, max_new_tokens=64, temperature=0.7, top_k=50, **kwargs):
        self.eval()
        for _ in range(max_new_tokens):
            idx_cond = idx[:, -self.config.max_seq_len:]
            logits, _ = self(idx_cond)
            logits = logits[:, -1, :] / temperature
            if top_k > 0:
                v, _ = torch.topk(logits, min(top_k, logits.size(-1)))
                logits[logits < v[:, [-1]]] = float("-inf")
            probs = F.softmax(logits, dim=-1)
            next_id = torch.multinomial(probs, num_samples=1)
            idx = torch.cat([idx, next_id], dim=1)
            if next_id.item() == self.config.eos_id:
                break
        return idx, []

    def param_count(self):
        total = sum(p.numel() for p in self.parameters())
        return total, 0

    def param_summary(self):
        total, _ = self.param_count()
        return f"MascotaLM: {total:,} params ({total/1e6:.1f}M)"


# Alias para compatibilidad
GuppyLM = MascotaLM


In [ ]:
%%writefile dataset.py
"""MascotaLM — carga del dataset."""

import json

import torch
from torch.utils.data import Dataset, DataLoader
from tokenizers import Tokenizer


class MascotaDataset(Dataset):
    def __init__(self, path: str, tokenizer_path: str, max_len: int = 512):
        self.tokenizer = Tokenizer.from_file(tokenizer_path)
        self.max_len = max_len
        self.samples = []

        with open(path, encoding='utf-8') as f:
            for line in f:
                data = json.loads(line)
                ids = self.tokenizer.encode(data["text"]).ids
                if len(ids) > max_len:
                    ids = ids[:max_len]
                if len(ids) >= 2:
                    self.samples.append(ids)

    def __len__(self):
        return len(self.samples)

    def __getitem__(self, idx):
        ids = self.samples[idx]
        x = ids[:-1]
        y = ids[1:]
        return torch.tensor(x, dtype=torch.long), torch.tensor(y, dtype=torch.long)


# Alias
GuppyDataset = MascotaDataset


def collate_fn(batch, pad_id=0):
    xs, ys = zip(*batch)
    max_len = max(len(x) for x in xs)
    padded_x = torch.full((len(xs), max_len), pad_id, dtype=torch.long)
    padded_y = torch.full((len(ys), max_len), pad_id, dtype=torch.long)
    for i, (x, y) in enumerate(zip(xs, ys)):
        padded_x[i, :len(x)] = x
        padded_y[i, :len(y)] = y
    return padded_x, padded_y


def get_dataloader(path, tokenizer_path, max_len=512, batch_size=32, shuffle=True):
    dataset = MascotaDataset(path, tokenizer_path, max_len)
    return DataLoader(
        dataset, batch_size=batch_size, shuffle=shuffle,
        collate_fn=collate_fn, num_workers=0, pin_memory=True,
    )


In [ ]:
%%writefile train.py
"""MascotaLM — loop de entrenamiento."""

import json
import math
import os
import time

import torch

from config import MascotaConfig, TrainConfig
from dataset import get_dataloader
from model import MascotaLM


def get_device(config):
    if config.device == "auto":
        if torch.cuda.is_available():
            return torch.device("cuda")
        if hasattr(torch.backends, "mps") and torch.backends.mps.is_available():
            return torch.device("mps")
        return torch.device("cpu")
    return torch.device(config.device)


def get_lr(step, config):
    if step < config.warmup_steps:
        return config.learning_rate * step / config.warmup_steps
    progress = (step - config.warmup_steps) / max(1, config.max_steps - config.warmup_steps)
    coeff = 0.5 * (1 + math.cos(math.pi * progress))
    return config.min_lr + (config.learning_rate - config.min_lr) * coeff


@torch.no_grad()
def evaluate(model, loader, device, max_batches=50):
    model.eval()
    total_loss, n = 0, 0
    for x, y in loader:
        if n >= max_batches:
            break
        x, y = x.to(device), y.to(device)
        _, loss = model(x, y)
        total_loss += loss.item()
        n += 1
    model.train()
    return total_loss / max(1, n)


def train():
    mc = MascotaConfig()
    tc = TrainConfig()
    device = get_device(tc)
    torch.manual_seed(tc.seed)

    print(f"Dispositivo: {device}")

    tokenizer_path = os.path.join(tc.data_dir, "tokenizer.json")
    model = MascotaLM(mc).to(device)
    print(model.param_summary())

    train_loader = get_dataloader(
        os.path.join(tc.data_dir, "train.jsonl"), tokenizer_path,
        mc.max_seq_len, tc.batch_size, shuffle=True,
    )
    eval_loader = get_dataloader(
        os.path.join(tc.data_dir, "eval.jsonl"), tokenizer_path,
        mc.max_seq_len, tc.batch_size, shuffle=False,
    )
    print(f"Train: {len(train_loader.dataset):,}, Eval: {len(eval_loader.dataset):,}")

    optimizer = torch.optim.AdamW(
        model.parameters(), lr=tc.learning_rate,
        weight_decay=tc.weight_decay, betas=(0.9, 0.95),
    )

    use_amp = device.type == "cuda"
    scaler = torch.amp.GradScaler("cuda") if use_amp else None

    os.makedirs(tc.output_dir, exist_ok=True)
    with open(os.path.join(tc.output_dir, "config.json"), "w") as f:
        json.dump({"model": vars(mc), "train": vars(tc)}, f, indent=2)

    model.train()
    step, best_eval = 0, float("inf")
    losses = []
    t0 = time.time()

    print(f"\nEntrenando por {tc.max_steps} pasos...")
    print(f"{'Paso':>6} | {'LR':>10} | {'Train':>10} | {'Eval':>10} | {'Tiempo':>8}")
    print("-" * 56)

    while step < tc.max_steps:
        for x, y in train_loader:
            if step >= tc.max_steps:
                break

            x, y = x.to(device), y.to(device)
            lr = get_lr(step, tc)
            for pg in optimizer.param_groups:
                pg["lr"] = lr

            if use_amp:
                with torch.amp.autocast("cuda"):
                    _, loss = model(x, y)
                scaler.scale(loss).backward()
                scaler.unscale_(optimizer)
                torch.nn.utils.clip_grad_norm_(model.parameters(), tc.grad_clip)
                scaler.step(optimizer)
                scaler.update()
            else:
                _, loss = model(x, y)
                loss.backward()
                torch.nn.utils.clip_grad_norm_(model.parameters(), tc.grad_clip)
                optimizer.step()

            optimizer.zero_grad(set_to_none=True)
            losses.append(loss.item())

            if step % 100 == 0:
                avg = sum(losses[-100:]) / len(losses[-100:])
                elapsed = time.time() - t0
                print(f"{step:6d} | {lr:10.6f} | {avg:10.4f} | {'--':>10} | {elapsed:7.1f}s")

            if step > 0 and step % tc.eval_interval == 0:
                el = evaluate(model, eval_loader, device)
                avg_train = sum(losses[-tc.eval_interval:]) / min(len(losses), tc.eval_interval)
                elapsed = time.time() - t0
                print(f"{step:6d} | {lr:10.6f} | {avg_train:10.4f} | {el:10.4f} | {elapsed:7.1f}s")

                if el < best_eval:
                    best_eval = el
                    torch.save({
                        "step": step,
                        "model_state_dict": model.state_dict(),
                        "config": vars(mc),
                        "eval_loss": el,
                    }, os.path.join(tc.output_dir, "best_model.pt"))
                    print(f"  -> Mejor modelo (eval={el:.4f})")

            if step > 0 and step % tc.save_interval == 0:
                torch.save({
                    "step": step,
                    "model_state_dict": model.state_dict(),
                    "config": vars(mc),
                }, os.path.join(tc.output_dir, f"step_{step}.pt"))

            step += 1

    torch.save({
        "step": step,
        "model_state_dict": model.state_dict(),
        "config": vars(mc),
        "train_losses": losses,
    }, os.path.join(tc.output_dir, "final_model.pt"))

    elapsed = time.time() - t0
    print(f"\n¡Listo! {elapsed:.0f}s, mejor eval: {best_eval:.4f}")


if __name__ == "__main__":
    train()


In [ ]:
%%writefile inference.py
"""MascotaLM — inferencia / chat con Mochi."""

import json
import os

import torch
from tokenizers import Tokenizer

from config import MascotaConfig
from model import MascotaLM


class MascotaInference:
    def __init__(self, checkpoint_path, tokenizer_path, device="cpu"):
        self.device = torch.device(device)
        self.tokenizer = Tokenizer.from_file(tokenizer_path)

        ckpt = torch.load(checkpoint_path, map_location=self.device, weights_only=False)

        # Cargar config.json desde el mismo directorio que el modelo
        config_dir = os.path.dirname(os.path.abspath(checkpoint_path))
        config_path = os.path.join(config_dir, "config.json")

        # Extraer state_dict — soporta formato legacy y estándar
        if isinstance(ckpt, dict) and "model_state_dict" in ckpt:
            state_dict = ckpt["model_state_dict"]
        else:
            state_dict = ckpt

        # Cargar config — config.json primero, luego embedded
        if os.path.exists(config_path):
            with open(config_path) as f:
                cfg = json.load(f)
            model_cfg = cfg.get("model", cfg)
            self.config = MascotaConfig(
                vocab_size=model_cfg.get("vocab_size", 4096),
                max_seq_len=model_cfg.get("max_seq_len", 128),
                d_model=model_cfg.get("d_model", 384),
                n_layers=model_cfg.get("n_layers", 6),
                n_heads=model_cfg.get("n_heads", 6),
                ffn_hidden=model_cfg.get("ffn_hidden", 768),
                dropout=model_cfg.get("dropout", 0.1),
            )
        elif isinstance(ckpt, dict) and "config" in ckpt:
            valid = set(MascotaConfig.__dataclass_fields__.keys())
            self.config = MascotaConfig(**{k: v for k, v in ckpt["config"].items() if k in valid})
        else:
            print("Advertencia: sin config, usando valores por defecto")
            self.config = MascotaConfig()

        self.model = MascotaLM(self.config).to(self.device)
        filtered = {k: v for k, v in state_dict.items() if k in self.model.state_dict()}
        self.model.load_state_dict(filtered)
        self.model.eval()

        total, _ = self.model.param_count()
        print(f"MascotaLM cargado: {total/1e6:.1f}M params")

    def chat_completion(self, messages, temperature=0.7, max_tokens=64, top_k=50, **kwargs):
        prompt = self._format_prompt(messages)
        input_ids = self.tokenizer.encode(prompt).ids
        prompt_tokens = len(input_ids)
        input_t = torch.tensor([input_ids], dtype=torch.long, device=self.device)

        output_t, _ = self.model.generate(input_t, max_tokens, temperature, top_k)
        output_text = self.tokenizer.decode(output_t[0].tolist()[prompt_tokens:])

        if "<|im_end|>" in output_text:
            output_text = output_text.split("<|im_end|>")[0]
        if "<|im_start|>" in output_text:
            output_text = output_text.split("<|im_start|>")[0]

        return {
            "choices": [{
                "message": {"role": "assistant", "content": output_text.strip()},
            }],
        }

    def _format_prompt(self, messages):
        parts = []
        for msg in messages:
            role = msg.get("role", "user")
            content = msg.get("content") or ""
            if role == "system":
                continue
            parts.append(f"<|im_start|>{role}\n{content}<|im_end|>")
        parts.append("<|im_start|>assistant\n")
        return "\n".join(parts)


# Alias para compatibilidad
GuppyInference = MascotaInference


def main():
    import argparse
    p = argparse.ArgumentParser(description="Chatear con Mochi")
    p.add_argument("--checkpoint", default="checkpoints/best_model.pt")
    p.add_argument("--tokenizer", default="data/tokenizer.json")
    p.add_argument("--device", default="cpu")
    args = p.parse_args()

    engine = MascotaInference(args.checkpoint, args.tokenizer, args.device)
    print("\nMochi Chat (escribí 'chau' para salir)")
    msgs = []
    while True:
        inp = input("\nVos> ").strip()
        if inp.lower() in ("chau", "salir", "exit", "q"):
            print("Mochi> bueno. chau. volvé pronto.")
            break
        msgs.append({"role": "user", "content": inp})
        result = engine.chat_completion(msgs)
        msg = result["choices"][0]["message"]
        if msg.get("content"):
            print(f"Mochi> {msg['content']}")
        msgs.append(msg)


if __name__ == "__main__":
    main()


## 3. Preparar Datos

Descargar el dataset de conversaciones desde HuggingFace y entrenar un tokenizer BPE.

El dataset tiene 60K conversaciones de turno único en 33 categorías:
saludos, comida, hambre, jugar, aburrimiento, cariño, elogios, abrazos,
estados emocionales, noche, mañana, nivel, sueños, clima, aventuras, y más.

Cada muestra está formateada como ChatML:
```
<|im_start|>user
hola mochi<|im_end|>
<|im_start|>assistant
oh hola. estaba pensando en comida. qué bueno que estás.<|im_end|>
```

In [ ]:
import json, os
from datasets import load_dataset
from tokenizers import Tokenizer, models, trainers, pre_tokenizers, decoders, processors

# ── Descargar desde HuggingFace ──
HF_DATASET = 'Brax055/Mascota_virtual'
ds = load_dataset(HF_DATASET)
print(f'Descargado: {len(ds["train"]):,} train, {len(ds["test"]):,} test samples')

# ── Formatear como ChatML y guardar como JSONL ──
os.makedirs('data', exist_ok=True)

def format_sample(row):
    return (
        f"<|im_start|>user\n{row['input']}<|im_end|>\n"
        f"<|im_start|>assistant\n{row['output']}<|im_end|>"
    )

for split, path in [('train', 'data/train.jsonl'), ('test', 'data/eval.jsonl')]:
    with open(path, 'w', encoding='utf-8') as f:
        for row in ds[split]:
            f.write(json.dumps({'text': format_sample(row), 'category': row['category']}, ensure_ascii=False) + '\n')
    print(f'Guardado: {path}')

# ── Entrenar tokenizer BPE ──
all_texts = [format_sample(row) for row in ds['train']]

tokenizer = Tokenizer(models.BPE(unk_token='<unk>'))
tokenizer.pre_tokenizer = pre_tokenizers.ByteLevel(add_prefix_space=False)
tokenizer.decoder = decoders.ByteLevel()

special_tokens = ['<pad>', '<|im_start|>', '<|im_end|>', '<unk>']
trainer = trainers.BpeTrainer(
    vocab_size=4096,
    special_tokens=special_tokens,
    min_frequency=2,
)
tokenizer.train_from_iterator(all_texts, trainer=trainer)
tokenizer.save('data/tokenizer.json')

enc = tokenizer.encode('hola mochi ¿cómo estás?')
print(f'\nTokenizer entrenado: {tokenizer.get_vocab_size()} tokens')
print(f'Test: {enc.tokens}')

## 4. Verificar Arquitectura

Sanity check rápido — construir el modelo, imprimir cantidad de parámetros, hacer un forward pass de prueba.

In [ ]:
from config import MascotaConfig
from model import MascotaLM
import torch

config = MascotaConfig()
model = MascotaLM(config)
print(model.param_summary())
print(f'  Capas: {config.n_layers}, Cabezas: {config.n_heads}, FFN: {config.ffn_hidden}')
print(f'  Vocab: {config.vocab_size}, Seq máx: {config.max_seq_len}')

# Forward pass de prueba
x = torch.randint(0, config.vocab_size, (2, 32))
logits, _ = model(x)
print(f'  Input shape: {x.shape} → Output shape: {logits.shape}')
print('\n¡Todo OK!')


## 5. Entrenar

10.000 pasos con schedule cosine de LR. Tarda ~2 min en T4.

El modelo aprende a:
- Responder en frases cortas y en minúsculas
- Mantenerse en personaje como Mochi, la mascota virtual
- Cubrir 33 categorías de conversación diferentes
- Dejar de generar en el momento correcto (aprende el token `<|im_end|>`)

In [ ]:
from train import train
train()

## 6. Probar

Chatear con el modelo entrenado. Cada mensaje es independiente (turno único).

In [ ]:
from inference import MascotaInference
import torch

engine = MascotaInference(
    'checkpoints/best_model.pt', 'data/tokenizer.json',
    device='cuda' if torch.cuda.is_available() else 'cpu'
)

def chat(prompt):
    r = engine.chat_completion([{'role': 'user', 'content': prompt}], max_tokens=64)
    return r['choices'][0]['message'].get('content', '').strip()

# Probar distintas categorías
tests = [
    'hola mochi',
    '¿tenés hambre?',
    '¿jugamos?',
    'contame un chiste',
    'hoy tuve que hacer trámites',
    'buenas noches mochi',
    'te quiero mucho',
    'volviste',
    'te traje un regalo',
]

for t in tests:
    print(f'Vos: {t}')
    print(f'Mochi: {chat(t)}')
    print()


## 7. Exportar y Subir a HuggingFace

Exportar el modelo en formato PyTorch y ONNX (cuantizado uint8, ~9 MB),
y subir todo a HuggingFace de una vez.

Configurá tu token y repo abajo.

In [ ]:
!pip install -q onnx onnxruntime onnxscript

from huggingface_hub import HfApi, login
import torch, json, os, shutil
from config import MascotaConfig
from model import MascotaLM

HF_TOKEN = os.environ.get('HF_TOKEN', '')   # O pegá tu token acá
HF_MODEL_REPO = os.environ.get('HF_MODEL_REPO', 'Brax055/mascotalm-9M')  # Repo del modelo

# Cargar checkpoint
ckpt = torch.load('checkpoints/best_model.pt', map_location='cpu')
config = MascotaConfig(**ckpt['config'])
model = MascotaLM(config)
model.load_state_dict(ckpt['model'])
model.eval()

# Guardar config
os.makedirs('hf_export', exist_ok=True)
with open('hf_export/config.json', 'w') as f:
    json.dump(ckpt['config'], f, indent=2)

# Guardar modelo PyTorch
torch.save(model.state_dict(), 'hf_export/pytorch_model.bin')
import shutil
shutil.copy('data/tokenizer.json', 'hf_export/tokenizer.json')

# Exportar ONNX cuantizado
from torch.onnx import export as onnx_export
import onnx
from onnxruntime.quantization import quantize_dynamic, QuantType

dummy = torch.randint(0, config.vocab_size, (1, 32))
onnx_export(model, dummy, 'hf_export/model_fp32.onnx',
            input_names=['input_ids'], output_names=['logits'],
            dynamic_axes={'input_ids': {0: 'batch', 1: 'seq'}},
            opset_version=17)
quantize_dynamic('hf_export/model_fp32.onnx', 'hf_export/model.onnx',
                 weight_type=QuantType.QUInt8)
os.remove('hf_export/model_fp32.onnx')

sz = os.path.getsize('hf_export/model.onnx') / 1e6
print(f'ONNX cuantizado: {sz:.1f} MB')

# Subir a HuggingFace
if HF_TOKEN:
    login(token=HF_TOKEN)
    api = HfApi()
    api.create_repo(HF_MODEL_REPO, exist_ok=True)
    for fname in os.listdir('hf_export'):
        api.upload_file(
            path_or_fileobj=f'hf_export/{fname}',
            path_in_repo=fname,
            repo_id=HF_MODEL_REPO,
        )
    print(f'\n¡Subido! https://huggingface.co/{HF_MODEL_REPO}')
else:
    print('Sin HF_TOKEN — archivos guardados en hf_export/ solamente.')

## 8. Descargar Localmente

O descargar el modelo como un archivo tar.gz.

In [ ]:
import os

!cd /content && tar czf mascotalm.tar.gz \
    mascota/checkpoints/best_model.pt \
    mascota/checkpoints/config.json \
    mascota/data/tokenizer.json \
    mascota/model.py \
    mascota/config.py \
    mascota/inference.py \
    mascota/hf_export/model.onnx

sz = os.path.getsize('/content/mascotalm.tar.gz') / 1e6
print(f'Paquete: /content/mascotalm.tar.gz ({sz:.1f} MB)')

try:
    from google.colab import files
    files.download('/content/mascotalm.tar.gz')
except:
    print('Descargá manualmente desde el panel de archivos de Colab.')